In [2]:
import os, sys, importlib, shutil, zipfile
import boto3, pycountry

import pandas as pd
import numpy as np

import GEP

importlib.reload(GEP)

<module 'GEP' from 'c:\\WBG\\Work\\Code\\gep_results_analysis\\GEP_analysis\\GEP.py'>

In [3]:
bucket = 'wbg-geography01'
prefix = '/GEP/GEP-Climate-2022'
region = 'us-east-1'
s3client = boto3.client('s3', region_name=region)

local_folder = 'C:/WBG/Work/GEP/Climate-2022/Data/Results'
results_folder = 'C:/WBG/Work/GEP/Climate-2022/Results'
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

In [4]:
# Download results from internal AWS bucket
### Not currently working on WB laptop
'''
# Loop through the S3 bucket and get all the file keys
more_results = True
try:
    del(token)
except:
    pass
loops = 0

all_res = []
while more_results:
    print(f"Completed loop: {loops}")
    if loops > 0:
        objects = s3client.list_objects_v2(Bucket=bucket, ContinuationToken=token, Prefix=prefix)
    else:
        objects = s3client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    more_results = objects['IsTruncated']
    if more_results:
        token = objects['NextContinuationToken']
    loops += 1
    for res in objects['Contents']:
        all_res.append(res)'''

'\n# Loop through the S3 bucket and get all the file keys\nmore_results = True\ntry:\n    del(token)\nexcept:\n    pass\nloops = 0\n\nall_res = []\nwhile more_results:\n    print(f"Completed loop: {loops}")\n    if loops > 0:\n        objects = s3client.list_objects_v2(Bucket=bucket, ContinuationToken=token, Prefix=prefix)\n    else:\n        objects = s3client.list_objects_v2(Bucket=bucket, Prefix=prefix)\n    more_results = objects[\'IsTruncated\']\n    if more_results:\n        token = objects[\'NextContinuationToken\']\n    loops += 1\n    for res in objects[\'Contents\']:\n        all_res.append(res)'

In [ ]:
# Unzip results folder
zip_files = [f for f in os.listdir(local_folder) if f.endswith('.zip')]
for z_file in zip_files:
    cur_out_folder = os.path.join(local_folder, z_file.split('.')[0])
    with zipfile.ZipFile(os.path.join(local_folder, z_file), 'r') as zip_ref:
        zip_ref.extractall(cur_out_folder)
    #os.remove(os.path.join(local_folder, z_file))

### Generating Summaries

In [5]:
resultsFolder = local_folder

In [6]:
# Create summary output table containing all the summary files stacked together
all_results = []
for root, dirs, files in os.walk(resultsFolder):
    for f in files:
        if "summary.csv" in f:
            all_results.append(os.path.join(root, f))

In [7]:
importlib.reload(GEP)
try:
    del(final)
except:
    pass
all_res = {}
broken_res = {}
for x in all_results:
    xx = GEP.gep_summary(x)
    tempD = xx.get_data()
    if tempD.shape[1] == 5:
        all_res[os.path.basename(x)[:16]] = tempD
    else:
        broken_res[x.split("/")[-1][:16]] = tempD
        

In [8]:
# concatenate all results together
final = pd.concat(all_res)
final = final.reset_index()

#Clean up columns
final.columns = ['lvl','lvl1','Attribute','2025','2030','Country','Scenario']
final['Attr'] = final['Attribute'].apply(lambda x: x[:1])
final['Country_name'] = final['Country'].apply(lambda x: pycountry.countries.get(alpha_2=x).name)
final = final.loc[final['Attr'] != 'M',]
final.drop(['lvl','lvl1'], axis=1, inplace=True)

# add details on tech type and pop summaries
final['Tech'] = final['Attribute'].apply(lambda x: GEP.get_simple_tech(x))
final['Tech_detail'] = final['Attribute'].apply(lambda x: GEP.get_tech(x))
final['2030Sum'] = final['2025'] + final['2030']
final['PP'] = final['Country'].apply(GEP.get_pp)
final.head()

,Attribute,2025,2030,Country,Scenario,Attr,Country_name,Tech,Tech_detail,2030Sum,PP
0,1.Population_Grid,2.463446e+07,3.810823e+07,ao,0_0_0_0_0_0,1,Angola,GRID,GRID,6.274269e+07,SAPP
1,1.Population_SA_Diesel,0.000000e+00,0.000000e+00,ao,0_0_0_0_0_0,1,Angola,SA,SA_Diesel,0.000000e+00,SAPP
2,1.Population_SA_PV,2.749204e+06,5.838852e+06,ao,0_0_0_0_0_0,1,Angola,SA,SA_PV,8.588056e+06,SAPP
3,1.Population_MG_Diesel,0.000000e+00,0.000000e+00,ao,0_0_0_0_0_0,1,Angola,MG,MG_Diesel,0.000000e+00,SAPP
4,1.Population_MG_PV,0.000000e+00,0.000000e+00,ao,0_0_0_0_0_0,1,Angola,MG,MG_PV,0.000000e+00,SAPP


In [7]:
# Summarize population served by technology, based on demand lever (first lever)
final_pop = final.loc[final['Attr'] == '1']
bottom_up_demand = '0_0_0_0_1_0'
low_demand       = '1_0_0_0_1_0'
high_demand      = '2_0_0_0_1_0'

bottom_d = final_pop.loc[final_pop['Scenario'].isin([bottom_up_demand, low_demand, high_demand]),]
pop_summary = pd.pivot_table(bottom_d, index='Country_name', columns=['Scenario','Tech'], values=['2025','2030Sum'], aggfunc=np.sum)
pop_summary.to_csv(os.path.join(results_folder, f'pop_summary_by_demand_{bottom_up_demand}.csv'))

C:\Users\WB411133\AppData\Local\Temp\ipykernel_15720\3463014502.py:8: FutureWarning: The provided callable <function sum at 0x0000019E82E1BA60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pop_summary = pd.pivot_table(bottom_d, index='Country_name', columns=['Scenario','Tech'], values=['2025','2030Sum'], aggfunc=np.sum)


# Summarize emissions
Attribute #5 is emissions

In [24]:
total_costs = final.loc[final['Attr'] == '4'].groupby(['Country_name','Scenario']).agg({'2025':np.sum, '2030Sum':np.sum}).reset_index()
total_costs

C:\Users\WB411133\AppData\Local\Temp\ipykernel_21540\4108859911.py:1: FutureWarning: The provided callable <function sum at 0x000002322BED9120> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  total_costs = final.loc[final['Attr'] == '4'].groupby(['Country_name','Scenario']).agg({'2025':np.sum, '2030Sum':np.sum}).reset_index()
C:\Users\WB411133\AppData\Local\Temp\ipykernel_21540\4108859911.py:1: FutureWarning: The provided callable <function sum at 0x000002322BED9120> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  total_costs = final.loc[final['Attr'] == '4'].groupby(['Country_name','Scenario']).agg({'2025':np.sum, '2030Sum':np.sum}).reset_index()


,Country_name,Scenario,2025,2030Sum
0,Angola,0_0_0_0_0_0,3.035180e+09,9.640674e+09
1,Angola,0_0_0_0_0_1,3.061050e+09,9.872950e+09
2,Angola,0_0_0_0_1_0,3.205218e+09,9.769660e+09
3,Angola,0_0_0_0_1_1,3.205218e+09,1.005890e+10
4,Angola,0_0_0_1_0_0,2.817233e+09,8.277959e+09
...,...,...,...,...
5337,Zimbabwe,2_1_1_0_1_1,2.152478e+09,7.488706e+09
5338,Zimbabwe,2_1_1_1_0_0,1.485430e+09,4.118818e+09
5339,Zimbabwe,2_1_1_1_0_1,1.487143e+09,5.532416e+09
5340,Zimbabwe,2_1_1_1_1_0,1.485430e+09,4.118818e+09


In [69]:
final_emissions = final.loc[final['Attr'] == '5']
all_res = []
for cntry, inD in final_emissions.groupby('Country_name'):
    emission_sum = inD.groupby('Scenario').agg({'2025':'sum', '2030Sum':'sum'}).reset_index().sort_values('2030Sum', ascending=False)
    try:
        default_scenario_lbl = '0_0_0_0_1_0'
        default_scenario = emission_sum.loc[emission_sum['Scenario'] == default_scenario_lbl].iloc[0]
    except:
        default_scenario_lbl = emission_sum['Scenario'].iloc[int(emission_sum.shape[0]/2)]
        default_scenario = emission_sum.loc[emission_sum['Scenario'] == default_scenario_lbl].iloc[0]
    emission_range = pd.DataFrame([emission_sum.iloc[0], default_scenario, emission_sum.iloc[-1]])
        
    low_emission_scenario = emission_range['Scenario'].iloc[-1]
    high_emission_scenario = emission_range['Scenario'].iloc[0]
    low_emission_cost = total_costs.loc[((total_costs['Country_name'] == cntry) & (total_costs['Scenario'] == low_emission_scenario)),'2030Sum'].iloc[0]
    default_emission_cost = total_costs.loc[((total_costs['Country_name'] == cntry) & (total_costs['Scenario'] == default_scenario_lbl)),'2030Sum'].iloc[0]
    high_emission_cost = total_costs.loc[((total_costs['Country_name'] == cntry) & (total_costs['Scenario'] == high_emission_scenario)),'2030Sum'].iloc[0]
    
    emission_range['COST_2030'] = [low_emission_cost, default_emission_cost, high_emission_cost]
    emission_range['Emission'] = ['high', 'default', 'low']
    emission_range['CNTRY'] = cntry
    all_res.append(emission_range.copy())

In [70]:
pd.concat(all_res)

,Scenario,2025,2030Sum,COST_2030,Emission,CNTRY
65,2_0_0_0_0_1,1.108708e+09,3.098833e+09,6.160694e+09,high,Angola
2,0_0_0_0_1_0,4.346359e+08,1.127411e+09,9.769660e+09,default,Angola
30,0_1_1_1_1_0,1.961082e+08,4.881659e+08,1.402397e+10,low,Angola
67,2_0_0_0_1_1,1.557527e+09,3.866190e+09,1.765097e+09,high,Bangladesh
2,0_0_0_0_1_0,1.318754e+09,3.329155e+09,5.417596e+09,default,Bangladesh
...,...,...,...,...,...,...
2,0_0_0_0_1_0,6.433051e+08,1.828590e+09,4.421782e+09,default,Zambia
28,0_1_1_1_0_0,2.385678e+07,5.952426e+07,1.153195e+10,low,Zambia
65,2_0_0_0_0_1,3.406411e+09,8.675431e+09,1.811650e+09,high,Zimbabwe
2,0_0_0_0_1_0,2.053813e+09,6.590936e+09,7.094711e+09,default,Zimbabwe
